In [ ]:
#r "nuget: ScottPlot, 5.0.*"
#r "../practice2026/task17/bin/Debug/net9.0/task17.dll"
using System;
using System.Diagnostics;
using System.IO;
using System.Collections.Generic;
using ScottPlot;
using task17;
using Microsoft.DotNet.Interactive.Formatting;

The below script needs to be able to find the current output cell; this is an easy method to get it.

Installed Packages ScottPlot, 5.0.56

Loading extensions from `C:\Users\Zver\.nuget\packages\skiasharp\2.88.9\interactive-extensions\dotnet\SkiaSharp.DotNet.Interactive.dll`


(6,7): error CS0246: Не удалось найти тип или имя пространства имен "task17" (возможно, отсутствует директива using или ссылка на сборку).



Error: compilation error

In [ ]:

Formatter.Register(typeof(ScottPlot.Plot), (plotObj, writer) =>
{
    var base64 = ((ScottPlot.Plot)plotObj).GetImageBytes(600, 400, ScottPlot.ImageFormat.Png);
    writer.Write($"<img src='data:image/png;base64,{Convert.ToBase64String(base64)}' width='600' height='400' />");
}, HtmlFormatter.MimeType);

int testRunsCount = 10; 

double CalculateMedian(double[] values)
{
    var sorted = values.OrderBy(v => v).ToArray();
    int mid = sorted.Length / 2;
    return sorted.Length % 2 == 0
        ? (sorted[mid - 1] + sorted[mid]) / 2.0
        : sorted[mid];
}


double MeasureSchedulerResponseDelay(int activeBackgroundTasks)
{
    var errorHandler = new SilentExceptionHandler();
    var server = new ServerThread(errorHandler);
    
    server.Start();

    long enqueueTimestampTicks = DateTime.UtcNow.Ticks;


    for (int k = 0; k < activeBackgroundTasks; k++)
    {
        server.QueueCommand(new HeavyBackgroundJob(slicesCount: 20));
    }


    var diagnosticPing = new DiagnosticPingCommand();
    server.QueueCommand(diagnosticPing);
    

    server.QueueCommand(new SoftStop(server));
    server.UnderlyingThread.Join();


    return TimeSpan.FromTicks(diagnosticPing.ExecutionTimestampTicks - enqueueTimestampTicks).TotalMilliseconds;
}


int[] backgroundLoadLevels = { 0, 2, 4, 8, 16, 32 };
double[] medianResponseTimes = new double[backgroundLoadLevels.Length];


for (int i = 0; i < backgroundLoadLevels.Length; i++)
{
    double[] measurements = new double[testRunsCount];
    for (int j = 0; j < testRunsCount; j++)
    {
        measurements[j] = MeasureSchedulerResponseDelay(backgroundLoadLevels[i]);
    }
    medianResponseTimes[i] = CalculateMedian(measurements);
}


string reportPath = "results.txt";
using (StreamWriter writer = new StreamWriter(reportPath))
{
    writer.WriteLine("АНАЛИЗ ВРЕМЕНИ ОТКЛИКА ПЛАНИРОВЩИКА ПРИ ФОНОВОЙ НАГРУЗКЕ (ROUND ROBIN)");
    writer.WriteLine($"{"Фоновые задачи (шт.)",-25} | {"Медианное время отклика (мс)",-30}");
    writer.WriteLine("---------------------------------------------------------------------");
    for (int i = 0; i < backgroundLoadLevels.Length; i++)
    {
        writer.WriteLine($"{backgroundLoadLevels[i],-25} | {medianResponseTimes[i],-30:F2} мс");
    }
}


var plt = new ScottPlot.Plot();
double[] xs = backgroundLoadLevels.Select(n => (double)n).ToArray();
var scatter = plt.Add.Scatter(xs, medianResponseTimes);

scatter.MarkerSize = 10;
scatter.LineWidth = 2.5f;
scatter.Color = new ScottPlot.Color(46, 139, 87); 

plt.XLabel("Уровень фоновой нагрузки (кол-во задач в очереди)");
plt.YLabel("Время отклика диагностической команды (мс)");
plt.Title("Показатели отзывчивости планировщика при многозадачности");

plt.SavePng("plot.png", 800, 600);


plt.Display();